# NumPy Advanced: Power Features

This notebook covers performance-critical and deep NumPy features used in production ML systems.

**Topics covered:**
1. Vectorization vs Loops
2. Strides & Memory Layout
3. Structured Arrays
4. Advanced Linear Algebra
5. Fourier Transforms (FFT)
6. Efficient Techniques
7. ML Context

In [ ]:
import numpy as np
import time

---
## 1. Vectorization vs Loops

**Vectorization** = replacing explicit Python loops with NumPy's optimized C-level operations. It's the #1 performance technique in NumPy.

In [ ]:
# Compute distances between two sets of points
np.random.seed(42)
n = 10000
A = np.random.rand(n, 3)  # n points in 3D
B = np.random.rand(n, 3)

In [ ]:
# Loop approach
start = time.time()
dists_loop = np.zeros(n)
for i in range(n):
    dists_loop[i] = np.sqrt(np.sum((A[i] - B[i]) ** 2))
loop_time = time.time() - start

# Vectorized approach
start = time.time()
dists_vec = np.sqrt(np.sum((A - B) ** 2, axis=1))
vec_time = time.time() - start

print(f"Loop:     {loop_time:.4f}s")
print(f"Vectorized: {vec_time:.6f}s")
print(f"Speedup:  {loop_time / vec_time:.1f}x faster")
print(f"Results match: {np.allclose(dists_loop, dists_vec)}")

In [ ]:
# np.frompyfunc — vectorize arbitrary Python functions
def classify(x):
    if x < 30:
        return "young"
    elif x < 60:
        return "middle"
    else:
        return "senior"

ages = np.array([25, 45, 70, 18, 55])
classify_vec = np.frompyfunc(classify, 1, 1)
print("Classifications:", classify_vec(ages))

---
## 2. Strides & Memory Layout

NumPy arrays are blocks of contiguous memory. **Strides** tell NumPy how to jump from one element to the next in each dimension.

In [ ]:
arr = np.array([[1, 2, 3],
                [4, 5, 6]], dtype=np.int32)

print("Shape:", arr.shape)
print("Strides:", arr.strides, "-> (bytes to jump to next row, next col)")
print("Dtype:", arr.dtype)
print("Itemsize:", arr.itemsize, "bytes")

# Strides = (columns * itemsize, itemsize) for C-order (row-major)
print(f"\nVerification: {arr.shape[1]} * {arr.itemsize} = {arr.shape[1] * arr.itemsize}")

In [ ]:
# Views vs Copies
original = np.array([1, 2, 3, 4, 5])
view = original[1:4]     # slice creates a VIEW
copy = original[1:4].copy()  # explicit COPY

view[0] = 99  # modifies original!
print("Original after modifying view:", original)  # [1, 99, 3, 4, 5]

copy[0] = 99  # does NOT modify original
print("Original after modifying copy:", original)   # unchanged

In [ ]:
# np.shares_memory — check if arrays share data
a = np.arange(12).reshape(3, 4)
b = a.T  # transpose is a view (no data copied)
c = a.copy()

print("a and a.T share memory?", np.shares_memory(a, b))  # True
print("a and a.copy() share memory?", np.shares_memory(a, c))  # False

In [ ]:
# C-order vs Fortran-order (column-major)
arr = np.arange(6).reshape(2, 3)

c_order = np.array(arr, order='C')    # row-major (default)
f_order = np.array(arr, order='F')    # column-major

print("C-order strides:", c_order.strides)
print("F-order strides:", f_order.strides)

# Column-major is faster for column-wise operations
big = np.random.rand(1000, 1000)

start = time.time()
_ = big.sum(axis=0)  # column-wise
print(f"\nColumn-wise sum (C-order): {time.time() - start:.5f}s")

big_f = np.asfortranarray(big)
start = time.time()
_ = big_f.sum(axis=0)  # column-wise
print(f"Column-wise sum (F-order): {time.time() - start:.5f}s")

---
## 3. Structured Arrays

Arrays with named fields and mixed data types — like a lightweight table without pandas.

In [ ]:
# Define a structured dtype
dt = np.dtype([
    ('name', 'U10'),      # Unicode string, max 10 chars
    ('age', 'i4'),        # 32-bit integer
    ('gpa', 'f8'),        # 64-bit float
])

# Create structured array
students = np.array([
    ('Alice', 20, 3.8),
    ('Bob', 22, 3.5),
    ('Charlie', 21, 3.9)
], dtype=dt)

print("Students:", students)
print("\nNames:", students['name'])
print("GPAs:", students['gpa'])
print("High GPA (>3.7):", students[students['gpa'] > 3.7]['name"])

In [ ]:
# Sorting by a field
sorted_students = np.sort(students, order='gpa')
print("Sorted by GPA:", sorted_students)

---
## 4. Advanced Linear Algebra

SVD, eigenvalues, and least squares — the math behind PCA, recommendation systems, and regression.

In [ ]:
# SVD (Singular Value Decomposition)
# A = U @ S @ Vt
np.random.seed(42)
A = np.random.rand(4, 3)

U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("U shape:", U.shape)
print("S (singular values):", S)
print("Vt shape:", Vt.shape)

# Reconstruct
A_reconstructed = U @ np.diag(S) @ Vt
print("\nReconstruction error:", np.allclose(A, A_reconstructed))

In [ ]:
# PCA sketch using SVD
np.random.seed(0)
X = np.random.randn(100, 3)  # 100 samples, 3 features
X_centered = X - X.mean(axis=0)

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Variance explained by each component
variance_explained = S ** 2 / np.sum(S ** 2)
print("Variance explained per component:", np.round(variance_explained, 3))
print("Cumulative:", np.round(np.cumsum(variance_explained), 3))

# Project onto first 2 components
X_projected = X_centered @ Vt[:2].T
print(f"\nProjected shape: {X_projected.shape} (100 samples, 2 components)")

In [ ]:
# Eigenvalues and eigenvectors
M = np.array([[4, 2],
              [1, 3]])

eigenvalues, eigenvectors = np.linalg.eig(M)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

# Verify: M @ v = lambda * v
for i in range(len(eigenvalues)):
    lhs = M @ eigenvectors[:, i]
    rhs = eigenvalues[i] * eigenvectors[:, i]
    print(f"\nEigenvector {i}: M@v = {lhs}, lambda*v = {rhs}")

In [ ]:
# Least squares (for overdetermined systems)
# Fit a line y = mx + b to noisy data
np.random.seed(42)
x = np.linspace(0, 10, 50)
y = 2.5 * x + 1.0 + np.random.randn(50) * 2  # y = 2.5x + 1 + noise

# Design matrix: [x, 1] for y = m*x + b
A = np.column_stack([x, np.ones_like(x)])
params, residuals, rank, s = np.linalg.lstsq(A, y, rcond=None)

print(f"Fitted line: y = {params[0]:.3f}x + {params[1]:.3f}")
print(f"True line:   y = 2.500x + 1.000")
print(f"Residual sum of squares: {residuals[0]:.2f}")

---
## 5. Fourier Transforms (FFT)

The Fast Fourier Transform decomposes signals into frequencies — used in signal processing, audio, and image compression.

In [ ]:
# Create a signal: two sine waves combined
np.random.seed(42)
fs = 1000  # sampling frequency
t = np.linspace(0, 1, fs)

# Signal: 50 Hz + 120 Hz + noise
signal = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
signal += 0.3 * np.random.randn(len(t))

print(f"Signal length: {len(t)} samples")
print(f"Duration: {1/fs * len(t):.1f} seconds")

In [ ]:
# Compute FFT
fft_vals = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(t), 1/fs)

# Magnitude spectrum (only positive frequencies)
positive = freqs > 0
magnitude = np.abs(fft_vals[positive])
freqs_pos = freqs[positive]

# Find dominant frequencies
top_indices = np.argsort(magnitude)[-4:][::-1]
print("Dominant frequencies:", freqs_pos[top_indices].round(1), "Hz")
print("Expected: 50 Hz and 120 Hz")

---
## 6. Efficient Techniques

Tips and tools for writing performant NumPy code.

In [ ]:
# np.einsum — Einstein summation for complex operations
A = np.random.rand(3, 4)
B = np.random.rand(4, 5)

# Matrix multiplication with einsum
C_einsum = np.einsum('ij,jk->ik', A, B)
C_matmul = A @ B
print("einsum == matmul?", np.allclose(C_einsum, C_matmul))

# Other useful patterns:
x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

print("\nDot product:", np.einsum('i,i->', x, y))           # 32
print("Outer product:\n", np.einsum('i,j->ij', x, y))       # 3x3 matrix
print("Element-wise:", np.einsum('i,i->i', x, y))           # [4, 10, 18]

In [ ]:
# np.fromfunction — create arrays from index functions
def distance_from_center(i, j):
    return np.sqrt((i - 5) ** 2 + (j - 5) ** 2)

grid = np.fromfunction(distance_from_center, (11, 11))
print("Distance from center (11x11 grid):")
print(np.round(grid, 1))

In [ ]:
# Memory-mapped files — work with datasets larger than RAM
# Create a large file on disk
filename = 'large_dataset.dat'
shape = (10000, 100)  # 1M elements

# Write data
data = np.random.randn(*shape)
data.tofile(filename)

# Memory-map it (loads only what you access)
mmapped = np.memmap(filename, dtype=np.float64, mode='r', shape=shape)
print(f"Shape: {mmapped.shape}")
print(f"First 5 rows:\n", mmapped[:5, :3])

# Clean up
del mmapped
import os
os.remove(filename)

In [ ]:
# np.unique with return_inverse — for encoding
categories = np.array(['cat', 'dog', 'cat', 'bird', 'dog', 'cat'])
unique_cats, inverse, counts = np.unique(categories, return_inverse=True, return_counts=True)

print("Unique categories:", unique_cats)
print("Integer encoding:", inverse)  # positions in unique array
print("Counts:", counts)

# Decode back
decoded = unique_cats[inverse]
print("Decoded:", decoded)

---
## 7. ML Context

Advanced NumPy patterns that appear in real ML systems.

In [ ]:
# Implementing softmax (used in classification)
def softmax(logits):
    # Subtract max for numerical stability
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum(axis=1, keepdims=True)

# Example: 3 classes, 5 samples
logits = np.array([[2.0, 1.0, 0.1],
                   [1.0, 3.0, 0.2],
                   [0.5, 0.5, 2.0],
                   [3.0, 0.1, 0.3],
                   [0.1, 0.1, 4.0]])

probs = softmax(logits)
print("Logits:\n", logits)
print("\nProbabilities (sum to 1):\n", np.round(probs, 3))
print("\nRow sums (should be 1):", probs.sum(axis=1).round(10))

In [ ]:
# Batch processing large datasets
np.random.seed(42)
n_samples = 10000
X = np.random.randn(n_samples, 64)  # 10K samples, 64 features

# Process in batches (like a neural network would)
batch_size = 256
batch_means = []

for i in range(0, n_samples, batch_size):
    batch = X[i:i+batch_size]
    batch_means.append(batch.mean(axis=0))

# Global mean from batch means
global_mean = np.mean(batch_means, axis=0)
direct_mean = X.mean(axis=0)

print(f"Batch-computed mean (first 5 features): {global_mean[:5].round(6)}")
print(f"Direct mean (first 5 features):         {direct_mean[:5].round(6)}")
print(f"Max difference: {np.abs(global_mean - direct_mean).max():.10f}")

In [ ]:
# K-means clustering step (core algorithm)
np.random.seed(42)
X = np.vstack([
    np.random.randn(50, 2) + [0, 0],
    np.random.randn(50, 2) + [5, 5],
    np.random.randn(50, 2) + [10, 0]
])

k = 3
centroids = X[np.random.choice(len(X), k, replace=False)]

# One step: assign points to nearest centroid
# Using broadcasting: (150, 1, 2) - (1, 3, 2) -> (150, 3, 2)
distances = np.sqrt(((X[:, None] - centroids[None, :]) ** 2).sum(axis=2))
labels = distances.argmin(axis=1)

print(f"Points assigned to clusters:")
for c in range(k):
    print(f"  Cluster {c}: {(labels == c).sum()} points")

In [ ]:
# Performance tips summary
print("""
NumPy Performance Tips:
========================
1. Prefer vectorized operations over Python loops
2. Use views (slicing) instead of copies when possible
3. Pre-allocate arrays with np.zeros/np.empty instead of appending
4. Avoid unnecessary type conversions
5. Use np.memmap for datasets larger than RAM
6. Choose the right axis for aggregations
7. Use np.einsum for complex multi-array operations
8. Set np.random.seed() for reproducibility
9. Profile with %timeit before optimizing
10. Consider Fortran order for column-heavy operations
""")

---
## Summary

You now know:
- Why vectorization beats loops and how to write vectorized code
- How strides and memory layout affect performance
- Structured arrays for mixed-type data
- SVD, eigenvalues, and least squares for ML math
- FFT for signal processing
- Einstein summation, memory-mapped files, and efficient patterns
- Real ML implementations: softmax, batch processing, K-means

**You've completed the NumPy tutorial series!** These notebooks form the foundation for ML libraries like pandas, scikit-learn, TensorFlow, and PyTorch.